# Day 10 — 다중검정과 FDR: 1,000번 검정하면 생기는 일

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-2/day-10.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-2/day-10.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 1,000개 가상 유전자: raw p-value, FDR, Bonferroni 비교

정답을 아는 가상 실험입니다. 유전자 1,000개 중 앞의 100개만 두 그룹 사이에 진짜 차이가 있고, 나머지 900개는 차이가 없습니다.

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(10)
n_genes, n_true, n = 1000, 100, 6      # 유전자 1000개 중 100개만 진짜 차이, 그룹당 6마리

effect = np.zeros(n_genes)
effect[:n_true] = 2.5                  # 앞의 100개 유전자만 진짜 차이(표준편차의 2.5배)
is_true = effect > 0

gc = rng.normal(0, 1, size=(n_genes, n))
flt = rng.normal(effect[:, None], 1, size=(n_genes, n))

pvals = stats.ttest_ind(flt, gc, axis=1).pvalue
reject, qvals, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")

def count(sig, label):
    tp = (sig & is_true).sum()         # 진짜 차이를 맞게 찾음
    fp = (sig & ~is_true).sum()        # 차이가 없는데 '유의'하다고 함
    print(f"{label}: 유의 {sig.sum()}개 = 진짜 {tp}개 + 가짜 {fp}개 "
          f"(가짜 비율 {fp / max(sig.sum(), 1):.0%})")

count(pvals < 0.05, "raw p < 0.05 ")
count(reject,       "FDR(BH) < 0.05")
count(pvals < 0.05 / n_genes, "Bonferroni    ")

### 예시 2 — BH 보정을 손으로 계산하기

In [ ]:
import numpy as np
from statsmodels.stats.multitest import multipletests

p = np.array([0.001, 0.008, 0.039, 0.041, 0.20])
m = len(p)
rank = np.arange(1, m + 1)                  # p가 이미 작은 순서로 정렬되어 있음
by_hand = p * m / rank                      # p × (전체 개수 / 순위)
by_hand = np.minimum.accumulate(by_hand[::-1])[::-1]   # 뒤에서부터 누적 최솟값
print("손으로 계산:", by_hand.round(3))
print("statsmodels:", multipletests(p, method="fdr_bh")[1].round(3))

### 예시 3 — 아무 차이도 없는 실험

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(3)
a = rng.normal(0, 1, size=(1000, 6))
b = rng.normal(0, 1, size=(1000, 6))      # 모든 유전자에서 진짜 차이 없음
p = stats.ttest_ind(a, b, axis=1).pvalue
print("raw p < 0.05:", (p < 0.05).sum(), "개")
print("FDR < 0.05  :", multipletests(p, method="fdr_bh")[0].sum(), "개")

## 직접 해보기

1. `notebooks/02_statistics_basics.ipynb` 맨 아래에 `## Day 10 — 다중검정과 FDR` 섹션을 만들고 예시 1을 실행합니다.
2. 같은 노트북에서 다음 코드로 그림 두 개를 나란히 그립니다. 왼쪽은 p-value histogram, 오른쪽은 작은 순으로 정렬한 p-value와 BH 합격선입니다.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist([pvals[~is_true], pvals[is_true]], bins=20, stacked=True,
             label=["no true difference", "true difference"])
axes[0].set_xlabel("raw p-value")
axes[0].set_ylabel("number of genes")
axes[0].legend()

order = np.argsort(pvals)
k = np.arange(1, n_genes + 1)
axes[1].plot(k[:200], pvals[order][:200], ".", label="sorted p-values")
axes[1].plot(k[:200], 0.05 * k[:200] / n_genes, label="BH line: 0.05 x rank / 1000")
axes[1].set_xlabel("rank")
axes[1].set_ylabel("p-value")
axes[1].legend()

plt.tight_layout()
plt.savefig("../figures/day10_fdr.png", dpi=150)
plt.show()